# Small-molecule embeddings

This notebook is the entity-level tutorial for drugs, metabolites, and other small molecules. It covers SMILES-based language models, RDKit fingerprints, GNN-style models, annotation, plotting, and model comparison through `BioEmbedder.embed(...)`.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, pl, tl

RUN_REAL_EMBEDDING = False
RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

embedder = BioEmbedder(device="auto", organism="human")

smiles = [
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # aspirin
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",  # ibuprofen
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # caffeine
    "CCOC(=O)C1=CC=CC=C1Cl",  # small ester
    "COC1=CC=C(C=C1)CCN",  # phenethylamine-like
    "CCN(CC)CCOC(=O)C1=CC=CC=C1",  # local anesthetic-like
]
compound_names = ["aspirin", "ibuprofen", "caffeine", "ester", "phenethylamine", "anesthetic_like"]


## 1. Generate molecular embeddings

Use one call for classical fingerprints or neural molecule models. Classical fingerprints are useful for fast local checks; ChemBERTa/MolFormer/GNN models are useful for learned chemistry spaces.


In [ ]:
if RUN_REAL_EMBEDDING:
    molecule_adata = embedder.embed(
        smiles,
        entity_type="molecule",
        id_type="smiles",
        model=["morgan_fp", "chemberta2MTR", "molformer_base"],
        output="anndata",
        harmonize_dim=64,
        missing="nan",
    )
    print(molecule_adata.obsm.keys())
else:
    print("Set RUN_REAL_EMBEDDING=True to compute molecular embeddings.")


## 2. Plot-ready molecule AnnData

Rows are molecules. Annotation columns can be computed from RDKit or fetched from ChEMBL/ChEBI/KEGG/PubChem through `tl.annotate_molecules(...)`.


In [ ]:
def make_molecule_demo(labels: list[str]) -> ad.AnnData:
    classes = pd.Series(
        ["NSAID", "NSAID", "xanthine", "ester", "amine", "anesthetic"],
        index=labels,
        name="class",
    )
    centers = {name: rng.normal(size=16) for name in classes.unique()}
    X_fp = np.vstack([centers[classes.loc[c]] + rng.normal(scale=0.22, size=16) for c in labels]).astype("float32")
    X_chemberta = (X_fp @ rng.normal(size=(16, 10)) + rng.normal(scale=0.3, size=(len(labels), 10))).astype("float32")
    X_molformer = (X_fp @ rng.normal(size=(16, 8)) + rng.normal(scale=0.35, size=(len(labels), 8))).astype("float32")

    obs = pd.DataFrame(
        {
            "compound": labels,
            "smiles": smiles,
            "class": classes.values,
            "mol_logp": [1.2, 3.5, -0.1, 2.1, 1.6, 2.7],
            "mol_qed": [0.55, 0.82, 0.43, 0.61, 0.48, 0.73],
            "mol_mw": [180, 206, 194, 198, 151, 235],
        },
        index=labels,
    )
    out = ad.AnnData(X=np.zeros((len(labels), 1), dtype="float32"), obs=obs, var=pd.DataFrame(index=["placeholder"]))
    out.obsm["X_morgan_fp"] = X_fp
    out.obsm["X_chemberta"] = X_chemberta
    out.obsm["X_molformer"] = X_molformer
    return out

molecule_space = make_molecule_demo(compound_names)
molecule_space


## 3. Annotate small molecules


In [ ]:
RUN_REMOTE_ANNOTATION = False

if RUN_REMOTE_ANNOTATION:
    molecule_space = tl.annotate_molecules(
        molecule_space,
        column="smiles",
        sources=["structural", "bioactivity", "pathways"],
        copy=True,
    )
else:
    print(molecule_space.obs[["compound", "class", "mol_logp", "mol_qed", "mol_mw"]])


## 4. Plot annotated molecule spaces


In [ ]:
pl.plot_embedding_space(
    molecule_space,
    obsm_key="X_morgan_fp",
    method="pca",
    color="class",
    annotate=True,
    annotate_col="compound",
    title="Morgan fingerprint space",
)

pl.embedding_color_panel(
    molecule_space,
    obsm_key="X_chemberta",
    method="pca",
    color_keys=["class", "mol_logp", "mol_qed"],
    annotate=True,
    annotate_col="compound",
)

pl.radar_chart(
    molecule_space,
    properties=["mol_logp", "mol_qed", "mol_mw"],
    group_by="class",
    title="Molecule property profiles by class",
)


## 5. Compare molecular embedding models


In [ ]:
_, mean_overlap = tl.compute_knn_overlap(molecule_space, "X_morgan_fp", "X_chemberta", k=2)
print(f"Mean Morgan/ChemBERTa KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta", "X_molformer"], k=2)
pl.cross_embedding_correlation(molecule_space, "X_morgan_fp", "X_molformer")
pl.embedding_norms(molecule_space, obsm_keys=["X_morgan_fp", "X_chemberta", "X_molformer"])


## 6. Save molecular embeddings


In [ ]:
if RUN_REAL_EMBEDDING:
    embedder.embed(
        smiles,
        entity_type="molecule",
        id_type="smiles",
        model=["morgan_fp", "chemberta2MTR"],
        output="table",
        fmt="zarr",
        path="molecule_embeddings.zarr",
        harmonize_dim=64,
    )
